In [25]:
# from google.colab import drive
# drive.mount('/content/drive')
# raiz="drive/MyDrive/proyecto_mineria/"
raiz=""

# Cargar datos y generar los conjuntos de entrenamiento y test

In [26]:
import pandas as pd
nombre_csv_logs="presentacion1_resultados.csv"
# file_path = 'data_set_limpio.pkl'
file_path = f'{raiz}datasets_pkl/data_set_limpio_sin_not_for_sale.pkl'

df = pd.read_pickle(file_path)

# Display the loaded DataFrame
print("shape: ",df.shape)
print(df.columns)
df.sample(n=5)


shape:  (186795, 70)
Index(['Nat', 'Division', 'Club', 'Based', 'Preferred Foot', 'Right Foot',
       'Left Foot', 'Position', 'Height', 'Weight', 'Age', 'Wage', 'AT Apps',
       'AT Gls', 'Team', 'Caps', 'Yth Apps', 'Style', 'Rc Injury', 'Best Role',
       'Best Duty', 'Best Pos', 'Acc', 'Aer', 'Agg', 'Agi', 'Ant', 'Bal',
       'Bra', 'Cmd', 'Com', 'Cmp', 'Cnt', 'Cor', 'Cro', 'Dec', 'Det', 'Dri',
       'Ecc', 'Fin', 'Fir', 'Fla', 'Fre', 'Han', 'Hea', 'Jum', 'Kic', 'Ldr',
       'Lon', 'L Th', 'Mar', 'Nat .1', 'OtB', '1v1', 'Pac', 'Pas', 'Pen',
       'Pos', 'Pun', 'Ref', 'TRO', 'Sta', 'Str', 'Tck', 'Tea', 'Tec', 'Thr',
       'Vis', 'Wor', 'transfer_value_estimado'],
      dtype='object')


,Nat,Division,Club,Based,Preferred Foot,Right Foot,Left Foot,Position,Height,Weight,Age,Wage,AT Apps,AT Gls,Team,Caps,Yth Apps,Style,Rc Injury,Best Role,Best Duty,Best Pos,Acc,Aer,Agg,Agi,Ant,Bal,Bra,Cmd,Com,Cmp,Cnt,Cor,Cro,Dec,Det,Dri,Ecc,Fin,Fir,Fla,Fre,Han,Hea,Jum,Kic,Ldr,Lon,L Th,Mar,Nat .1,OtB,1v1,Pac,Pas,Pen,Pos,Pun,Ref,TRO,Sta,Str,Tck,Tea,Tec,Thr,Vis,Wor,transfer_value_estimado
17867,CHI,Chilean Second Division,Deportes Iberia,Chile (Second Division),Left,Reasonable,Very Strong,D (LC),181,75,20,400,41,1,-,0,0,Physical,-,Full-Back,Defend,D (L),14,1,5,12,10,8,9,1,1,8,8,4,2,13,5,6,1,7,8,5,6,2,14,9,1,13,7,7,11,12,6,1,9,8,6,8,3,1,3,7,7,5,8,8,2,7,8,80000
36403,CHI,-,-,Chile,Right,Very Strong,Reasonable,AM (RL),179,68,21,0,0,0,-,0,0,Technical,-,Winger,Support,AM (R),13,2,6,11,9,6,6,3,2,3,5,7,11,9,5,10,2,4,10,9,5,2,5,3,2,11,7,5,5,17,10,2,11,6,2,5,2,1,2,7,3,7,7,9,4,10,11,0
104319,ITA,Italian Serie B,Pisa,Italy (Serie B),Right,Very Strong,Weak,D (C),180,68,17,950,0,0,-,0,0,Intelligent,-,No-Nonsense Centre-Back,Defend,D (C),7,1,7,5,10,4,6,3,2,1,3,2,1,12,7,1,1,1,4,5,1,4,10,8,3,1,3,1,12,13,3,3,7,4,1,12,1,2,3,7,4,10,3,1,4,3,4,30500
123590,TUR,Turkish 2. League White Group,Afyonspor,Turkey (2. League White Group),Right,Very Strong,Weak,GK,188,74,25,350,1,0,-,0,0,Shot Stopper,-,Goalkeeper,Defend,GK,10,11,9,13,7,3,11,8,7,3,4,2,1,8,7,1,6,1,1,2,4,11,1,10,6,8,2,1,3,13,2,3,10,3,1,7,8,11,8,2,2,1,3,1,5,2,2,0
107370,TUR,Turkish 2. League Red Group,1461 Trabzon FK,Turkey (2. League Red Group),Right,Very Strong,Reasonable,"M (C), AM (RLC)",175,68,23,3500,54,6,-,0,0,Technical,-,Inverted Winger,Support,AM (L),13,1,10,14,9,8,7,1,1,11,8,10,11,8,11,14,1,10,13,13,10,2,7,4,3,4,11,6,4,12,11,3,12,12,8,7,2,1,3,10,4,6,11,14,3,12,10,230000


# Generar conjuntos de entrenamiento y test

In [27]:
import pandas as pd
import json
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def log_results(model, method_name, X_train, y_train, X_test, y_test, filepath=f"{raiz}results_regressionCV.csv"):
    # parameters
    params_dict = model.get_params()
    
    # Convert non-serializable objects
    for k, v in params_dict.items():
        try:
            json.dumps(v)
        except TypeError:
            params_dict[k] = str(v)

    params = json.dumps(params_dict)
    # Predictions
    y_pred = model.predict(X_test)
    # Metrics
    r2_train = model.score(X_train, y_train)
    r2_test = model.score(X_test, y_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    # Row to store
    row = {
        "method": method_name,
        "hyperparameters": json.dumps(params),
        "r2_train": r2_train,
        "r2_test": r2_test,
        "mae": mae,
        "rmse": rmse
    }
    # Append or create CSV
    try:
        df = pd.read_csv(filepath)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    except FileNotFoundError:
        df = pd.DataFrame([row])
    print("si pase")
    df.to_csv(filepath, index=False)

# Generar conjuntos de entrenamiento y test

In [28]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [29]:
import clustering as cl

kmeans_model_club = cl.fit_kmeans(train_df, columna="Club")
kmeans_model_nat = cl.fit_kmeans(train_df, columna="Nat")
kmeans_model_division = cl.fit_kmeans(train_df, columna="Division")


train_df = cl.apply_kmeans(train_df, kmeans_model_club)
train_df = cl.apply_kmeans(train_df, kmeans_model_nat)
train_df = cl.apply_kmeans(train_df, kmeans_model_division)

test_df = cl.apply_kmeans(test_df, kmeans_model_club)
test_df = cl.apply_kmeans(test_df, kmeans_model_nat)
test_df = cl.apply_kmeans(test_df, kmeans_model_division)



# Considerar solo las más importantes
Si previamente ejecutamos el algoritmo y ya tenemos el archivo

In [ ]:
import json
target="transfer_value_estimado"
with open("importantes.json", "r") as f:
    cols_to_keep = json.load(f)
    

train_df=train_df[cols_to_keep]
test_df=train_df[cols_to_keep]

In [ ]:

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

KeyError: "['transfer_value_estimado'] not found in axis"

## One Hot para las features categóricas

In [ ]:
# import numpy as np
# # categorical_cols=["Nat_cluster","Division_cluster","Club_cluster","Preferred Foot","Right Foot","Left Foot","Best Pos","Best Duty","Style","Best Role","Rc Injury"]
# categorical_cols=["Nat_cluster","Division_cluster","Best Duty","Style"]
# X_train = pd.get_dummies(X_train, columns=categorical_cols)
# X_test  = pd.get_dummies(X_test, columns=categorical_cols)

# X_train, X_test = X_train.align(X_test, join='left', axis=1,fill_value=0)

In [ ]:
# X_train = X_train.select_dtypes(include=[np.number,np.bool_])
# X_test = X_test.select_dtypes(include=[np.number,np.bool_])

In [ ]:
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()

# X_train = pd.DataFrame(
#     scaler.fit_transform(X_train),
#     columns=X_train.columns,
#     index=X_train.index
# )

# X_test = pd.DataFrame(
#     scaler.transform(X_test),
#     columns=X_test.columns,
#     index=X_test.index
# )

# Entrenamiento

In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Identify categorical columns
cat_cols = X_train.select_dtypes(exclude=['number']).columns.tolist()

print(f"Categorical columns: {cat_cols}")

# 2. Copy datasets
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

# 3. Encode all categorical columns
encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

X_train_encoded[cat_cols] = encoder.fit_transform(
    X_train[cat_cols]
)

X_test_encoded[cat_cols] = encoder.transform(
    X_test[cat_cols]
)

# 4. Train model
model = HistGradientBoostingRegressor()

# 

Categorical columns: ['Nat', 'Division', 'Club', 'Based', 'Preferred Foot', 'Right Foot', 'Left Foot', 'Position', 'Team', 'Style', 'Rc Injury', 'Best Role', 'Best Duty', 'Best Pos']


# Entrenamiento y Evaluación

In [ ]:
model.fit(X_train_encoded, y_train)
# model.fit(X_train_encoded, y_train)

# # 5. Predict
preds = model.predict(X_test_encoded)
log_results(model,"HistGradientBoostingRegressor",X_train_encoded,y_train,X_test_encoded,y_test,filepath=nombre_csv_logs)

si pase


In [ ]:
preds = model.predict(X_test_encoded)
print("saco ",model.score(X_test_encoded,y_test))
preds

saco  0.6567138327387729


array([494608.69951264, -37183.94102769,   6631.47804903, ...,
         1605.7919201 ,  -2412.70969382,  24263.9760251 ], shape=(37359,))

# Feature importance

In [ ]:
from sklearn.inspection import permutation_importance
import pandas as pd

result = permutation_importance(
    model,          # trained HistGradientBoostingRegressor
    X_test_encoded,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="neg_mean_absolute_error"
)

importance_df = pd.DataFrame({
    "feature": X_test_encoded.columns,
    "importance_mean": result.importances_mean,
    "importance_std": result.importances_std
})

importance_df = importance_df.sort_values(
    "importance_mean",
    ascending=False
)

print(importance_df)

   feature  importance_mean  importance_std
11    Wage    392913.997834     4314.526063
10     Age    238141.405204     5987.515742
3    Based     19447.167435     1968.517242
2     Club      9003.171223     2572.661463
22     Acc      7054.597496     1104.570043
..     ...              ...             ...
39     Fin      -944.418677      355.838883
51  Nat .1     -1371.515770      469.211969
36     Det     -1392.117486      568.215394
52     OtB     -2216.496939      426.566029
32     Cnt     -3567.014372      648.981387

[72 rows x 3 columns]


In [ ]:
import pandas as pd
visualizar_todo=True
if visualizar_todo:
    # Show all rows
    pd.set_option('display.max_rows', None)

    # Show all columns
    pd.set_option('display.max_columns', None)

    # Show full content of each cell (prevent text truncation)
    pd.set_option('display.max_colwidth', None)

print(importance_df)

             feature  importance_mean  importance_std
11              Wage    392913.997834     4314.526063
10               Age    238141.405204     5987.515742
3              Based     19447.167435     1968.517242
2               Club      9003.171223     2572.661463
22               Acc      7054.597496     1104.570043
15              Caps      7004.287088      285.602005
54               Pac      4376.331002     1299.590796
69      Club_cluster      4243.495964      124.015271
31               Cmp      3076.852093     1137.857500
35               Dec      2898.003436      617.670706
62               Str      2786.561579      318.281380
14              Team      1851.430937     1098.112352
27               Bal      1695.641355      245.856993
37               Dri      1529.041660      401.571053
0                Nat      1524.131459      708.352237
13            AT Gls      1194.457045      276.765319
65               Tec      1047.548120      661.568178
64               Tea       9

In [ ]:
genera_json=False
if genera_json:
    importantes=importance_df.loc[importance_df["importance_mean"] >= 0, "feature"].unique()
    importantes=importantes.tolist()
    import json

    with open("importantes.json", "w") as f:
        json.dump(importantes, f)


# Grid search

In [ ]:
# from sklearn.model_selection import GridSearchCV
# param_grid = {
#     'regressor__max_iter': [100, 200],
#     'regressor__learning_rate': [0.01, 0.1],
#     'regressor__max_leaf_nodes': [31, 63],
#     # You can even tune the preprocessor!
#     'prep__high_card__smooth': ['auto', 1.0] 
# }
# gbr_cv = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, n_jobs=-1, scoring='neg_mean_squared_error')
# gbr_cv.fit(X_train, y_train)

## Evaluación

In [ ]:
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# y_pred = gbr_cv.predict(X_test)
# mae=mean_absolute_error(y_test, y_pred)
# mse=mean_squared_error(y_test, y_pred)
# r2s=r2_score(y_test, y_pred)

# print("Best parameters ",gbr_cv.best_params_)
# print("mean_absolute_error: ",mae)
# print("mean_squared_error: ",mse)
# print("r2_score: ",r2s)